1. 앞서 작성한 `parser.py`의 함수들을 이용해서 pdf, ppt, excel, word 파일을 parsing해서 준비해 봅시다.

2. `chunk_splitter`를 사용해 chunk로 분할해 봅시다!

3. 그리고 모든 chunk를 합쳐 `total_chunks`라는 이름의 변수를 만들어 봅시다.

In [ ]:
from parsing import *

word_docs = parse_word("./data/키키테크_사내규정_행동강령.docx")
pdf_docs = parse_pdf("./data/키키테크_AI솔루션_제품카탈로그.pdf")
excel_docs = parse_excel("./data/키키테크_임직원및프로젝트현황.xlsx")
ppt_docs = parse_pptx("./data/키키테크_회사소개.pptx")

word_chunks = chunk_splitter(word_docs)
pdf_chunks = chunk_splitter(pdf_docs)
ppt_chunks = chunk_splitter(ppt_docs)
excel_chunks = chunk_splitter(excel_docs)

total_chunks = word_chunks + pdf_chunks + ppt_chunks + excel_chunks

In [4]:
print(len(total_chunks))

124


### 2. 텍스트 벡터로 변환

이제 llm이 이해할 수 있는 숫자 벡터 형태로 텍스트를 변환해 줘야 합니다.

그 방식에는 2가지가 있습니다.

1. Sparse Retriever (BM25) : 두 문서에 공통적으로 등장한 단어가 얼마나 많은 지를 비교합니다.
2. Dense Retriever (DL) : 딥러닝 모델을 이용해 문장의 내제적 의미를 담은 벡터를 서로 비교합니다.

Sparse Retriever에서 가장 많이 사용되는 알고리즘 중 하나인 BM25입니다.

langchain에서 기본적으로 제공해 주기도 합니다.

In [5]:
from langchain_community.retrievers import BM25Retriever

# BM25 sparse retriever 생성 (rank_bm25 패키지 필요: pip install rank-bm25)
bm25_retriever = BM25Retriever.from_documents(total_chunks)
bm25_retriever.k = 3  # 상위 3개 문서 반환

# 테스트
results = bm25_retriever.invoke("키키테크의 주요 수익모델은 뭐야?")
for i, doc in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print(f"출처: {doc.metadata.get('source', '엑셀 데이터')}")
    print(f"내용: {doc.page_content[:200]}")


--- Result 1 ---
출처: ./data/키키테크_AI솔루션_제품카탈로그.pdf
내용: 9. 가격 정책 및 라이선스
키키테크의 모든 제품은 구독형(SaaS)과 영구 라이선스(온프레미스) 두 가지 방식으로
제공됩니다. 아래 가격은 기준 가격이며, 도입 규모, 계약 기간, 번들 구성에 따라 협의
할인이 가능합니다.
TalkBridge Enterprise 가격표
 플랜
 월 구독료
 포함 사용자
 문서 한도
 Starter
 월 200만원
 50명

--- Result 2 ---
출처: ./data/키키테크_사내규정_행동강령.docx
내용: 1.2 핵심 가치 (Core Values)

혁신 (Innovation): 새로운 기술과 아이디어를 두려움 없이 탐구하고 도전합니다.

신뢰 (Trust): 고객, 동료, 파트너와의 관계에서 언제나 투명하고 책임감 있게 행동합니다.

협력 (Collaboration): 다양한 배경과 역량을 가진 구성원들이 함께 더 큰 성과를 만들어냅니다.

성장 (Grow

--- Result 3 ---
출처: ./data/키키테크_AI솔루션_제품카탈로그.pdf
내용: 3. DataSight AI — 지능형 데이터 분석 플랫폼
3.1 제품 개요
DataSight AI는 비전문가도 자연어로 데이터를 분석하고 시각화할 수 있는 지능형 데이터 분석
플랫폼입니다. '지난 분기 지역별 매출 상위 5개 제품을 보여줘'와 같은 자연어 질의만으로
복잡한 SQL 쿼리와 시각화 차트를 자동으로 생성합니다.
3.2 주요 기능
 NL2SQL


Dense Retriever를 만들기 위해선 딥러닝 모델이 필요합니다.

여기에는 텍스트 생성 모델(DS Assistant)이 아니라 텍스트 임베딩에 특화된 언어 모델이 사용됩니다.

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_core.embeddings import Embeddings
from dotenv import load_dotenv
import requests
import os

total_contents = excel_contents + word_chunks + pdf_chunks + ppt_contents

load_dotenv(dotenv_path="../../.env")

embedding_api_url = os.getenv("embedding_api_url")
embedding_x_dep_ticket = os.getenv("embedding_x_dep_ticket")

# 1. API를 통해 임베딩을 생성하는 커스텀 클래스 정의
class MyCustomEmbeddings(Embeddings):
    def __init__(self, api_url, ticket):
        self.api_url = api_url
        self.ticket = ticket
    
    def embed_documents(self, texts):
        response = requests.post(
            f"{self.api_url}/v1/embeddings",
            json={"input": texts},
            headers={"Content-Type": "application/json", "x-dep-ticket": self.ticket}
        )
        return [item["embedding"] for item in response.json()['data']]

    def embed_query(self, text):
        return self.embed_documents([text])[0]

# 2. 임베딩 모델 객체 생성
custom_embedding_model = MyCustomEmbeddings(embedding_api_url, embedding_x_dep_ticket)

# 3. 데이터 준비
page_contents = [content.page_content for content in total_contents]
page_metadatas = [content.metadata for content in total_contents]

embeddings = custom_embedding_model.embed_documents(page_contents)

#### FAISS란?

의미가 비슷한 것끼리 빠르게 찾아주는 도서관 사서와 같은 역할을 합니다.

벡터 형태를 빠르게 찾을 수 있도록 정렬해서 보관(vectorstore)해 준다고 생각하시면 됩니다!

In [ ]:
# 4. FAISS 벡터스토어 생성
text_embeddings = list(zip(page_contents, embeddings))
vectorstore = FAISS.from_embeddings(text_embeddings, custom_embedding_model)

벡터는 이렇게 생겼습니다.

In [ ]:
vectors = np.zeros((vectorstore.index.ntotal, vectorstore.index.d), dtype=np.float32)
vectorstore.index.reconstruct_n(0, vectorstore.index.ntotal, vectors)

print(vectors.shape)
print(vectors[0][:10])

`vectorstore.similarity_search()`를 통해 빠르게 유사 문서를 찾을 수 있습니다.

In [ ]:
results = vectorstore.similarity_search("키키테크의 주요 수익 모델은?", k=3)

for i, doc in enumerate(results, 1):
    print(f"출처: {doc.metadata.get("source")}")
    print(f"내용: {doc.page_content[:200]}")

생성된 벡터는 저장할 수 있습니다.

In [ ]:
vectorstore.save_local("faiss_index")

아래 코드로 다시 불러오는 것도 가능합니다.

In [ ]:
vectorstore = FAISS.load_local(
    "faiss_index",
    custom_embedding_model,
    allow_dangerous_deserialization=True
)

### Ensemble Retriever

일반적으로 retriever에는 sparse retriever와 dense retriever를 일정 비율로 섞어서 관련 문서를 찾아옵니다.

`EnsembleRetriever`는 두 retriever의 결과를 **RRF(Reciprocal Rank Fusion)** 알고리즘으로 합산합니다.

| 방식 | 장점 | 단점 |
|------|------|------|
| **Sparse (BM25)** | 고유명사·숫자 등 키워드 정확 매칭 | 의미 유사도 파악 불가 |
| **Dense (FAISS)** | 문맥·의미 기반 유사도 검색 | 정확한 키워드 매칭 약함 |
| **Hybrid (Ensemble)** | 두 방식의 장점 보완 | — |

`weights`로 각 retriever의 기여 비율을 조절할 수 있으며, 합이 1이 되어야 합니다.

In [ ]:
from langchain_classic.retrievers import EnsembleRetriever

# FAISS dense retriever
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Sparse(BM25) 40% + Dense(FAISS) 60% 비율로 혼합
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, dense_retriever],
    weights=[0.4, 0.6]
)

# 테스트: 동일한 질문으로 하이브리드 검색 결과 확인
results = ensemble_retriever.invoke("키키테크의 주요 수익모델은 뭐야?")
for i, doc in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print(f"출처: {doc.metadata.get('source', '엑셀 데이터')}")
    print(f"내용: {doc.page_content[:200]}")

#### 실습 1 - rag_retriever.py

앞서 살펴본 파싱 함수들과 EnsembleRetriever 생성 과정을 하나의 클래스로 묶어 봅시다!

클래스 이름은 `RAGRetriever`로 하고, 아래 기능을 구현해 보세요.

1. `__init__` : 임베딩 모델과 splitter를 초기화합니다.
2. `parse_excel` / `parse_pdf` / `parse_word` / `parse_pptx` : 각 파일 형식을 Document 리스트로 변환합니다.
3. `add_file(file_path)` : 파일 확장자를 보고 알맞은 파싱 함수를 호출한 뒤, chunk로 분할해 `total_chunks`에 추가합니다.
4. `build_retriever(sparse_weight, dense_weight, k)` : `total_chunks`를 바탕으로 BM25와 FAISS retriever를 만들고 `EnsembleRetriever`로 합칩니다.
5. `search(query)` : `ensemble_retriever`로 관련 문서를 검색해 반환합니다.

#### 실습 2 - retriever_tool

이제 실습 1에서 만든 `RAGRetriever`를 활용해서 agent가 사용할 수 있는 tool을 만들어 봅시다!

순서는 다음과 같습니다.

1. `RAGRetriever` 인스턴스를 만들고 `add_file()`로 아래 4개 파일을 추가하세요.
   - `data/키키테크_임직원및프로젝트현황.xlsx`
   - `data/키키테크_AI솔루션_제품카탈로그.pdf`
   - `data/키키테크_사내규정_행동강령.docx`
   - `data/키키테크_회사소개.pptx`

2. `build_retriever()`를 호출해 EnsembleRetriever를 생성하세요.

3. `@tool` 데코레이터를 사용해 `search_documents(query)` 함수를 작성하세요.
   - 함수 내부에서 위에서 만든 인스턴스의 `search()`를 호출합니다.
   - 검색 결과를 `[출처: 파일명]\n내용` 형태의 문자열로 만들어 반환합니다.
   - docstring에 tool의 역할을 설명해 주세요. (agent는 docstring을 보고 tool 사용 여부를 결정합니다!)